In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
from pathlib import Path
import sys
import os

ROOT_DIR = Path().resolve().parent
sys.path.insert(0, str(ROOT_DIR))

from scripts.datasets import load_taxi_data

In [ ]:
df = load_taxi_data()

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6593828 entries, 0 to 6593827
Data columns (total 26 columns):
 #   Column                      Dtype                          
---  ------                      -----                          
 0   trip_id                     str                            
 1   taxi_id                     str                            
 2   trip_start_timestamp        datetime64[us, America/Chicago]
 3   trip_end_timestamp          datetime64[us, America/Chicago]
 4   trip_seconds                int64                          
 5   trip_miles                  float64                        
 6   pickup_census_tract         int64                          
 7   dropoff_census_tract        int64                          
 8   pickup_community_area       float64                        
 9   dropoff_community_area      float64                        
 10  fare                        float64                        
 11  tips                        float64             

In [4]:
df.head()

,trip_id,taxi_id,trip_start_timestamp,trip_end_timestamp,trip_seconds,trip_miles,pickup_census_tract,dropoff_census_tract,pickup_community_area,dropoff_community_area,...,company,pickup_centroid_latitude,pickup_centroid_longitude,dropoff_centroid_latitude,dropoff_centroid_longitude,hour,dayofweek,month,trip_minutes,tip_pct
0,22208de57c456e7e6ea5f60bdc1746ad300535a9,04b96cbbdcfe5b7cbb6884bc1b922819466f652662ead8...,2024-01-01 00:00:00-06:00,2024-01-01 00:30:00-06:00,2214,18.23,17031980000,17031320100,76.0,32.0,...,5 Star Taxi,41.979071,-87.903040,41.884987,-87.620993,0.0,Monday,2024-01,36.900000,16.393782
1,35968e44a8ea32a0849720b91c35a4d5a8ff6484,4a991432c3e0600b9c919a01148b17b866d29a41751b95...,2024-01-01 00:00:00-06:00,2024-01-01 00:00:00-06:00,120,0.00,17031081600,17031081201,8.0,8.0,...,Taxi Affiliation Services,41.892073,-87.628874,41.899156,-87.626211,0.0,Monday,2024-01,2.000000,0.0
2,3c05ccf0732fc338b7c875f9a9779039eaada274,0cbf5c0f6aca3628d77c7b6fe89715757ed402a70b0f8b...,2024-01-01 00:00:00-06:00,2024-01-01 00:30:00-06:00,1681,15.34,17031980000,17031071400,76.0,7.0,...,Globe Taxi,41.979071,-87.903040,41.922083,-87.634156,0.0,Monday,2024-01,28.016667,22.264151
3,541cf2b862280d13b36e466ad90d9485e1ae1600,13c8f729e7e5a9f850e406e3b31524c6881649044dab76...,2024-01-01 00:00:00-06:00,2024-01-01 00:00:00-06:00,59,0.00,17031980000,17031980000,76.0,76.0,...,5 Star Taxi,41.979071,-87.903040,41.979071,-87.903040,0.0,Monday,2024-01,0.983333,0.0
4,63d8c865c01bde9e17e469db6a30e33c8cfe5314,259d38cfdbc9ac6f9bb01f0df740e0ddf4a631a70bbdd6...,2024-01-01 00:00:00-06:00,2024-01-01 00:00:00-06:00,180,0.30,17031081500,17031081201,8.0,8.0,...,"Taxicab Insurance Agency, LLC",41.892508,-87.626215,41.899156,-87.626211,0.0,Monday,2024-01,3.000000,0.0


In [6]:
df.columns

Index(['trip_id', 'taxi_id', 'trip_start_timestamp', 'trip_end_timestamp',
       'trip_seconds', 'trip_miles', 'pickup_census_tract',
       'dropoff_census_tract', 'pickup_community_area',
       'dropoff_community_area', 'fare', 'tips', 'tolls', 'extras',
       'trip_total', 'payment_type', 'company', 'pickup_centroid_latitude',
       'pickup_centroid_longitude', 'dropoff_centroid_latitude',
       'dropoff_centroid_longitude', 'hour', 'dayofweek', 'month',
       'trip_minutes', 'tip_pct'],
      dtype='str')

## Handle missing values




In [8]:
print(df.isnull().sum())

trip_id                           0
taxi_id                           4
trip_start_timestamp            104
trip_end_timestamp              111
trip_seconds                      0
trip_miles                        0
pickup_census_tract               0
dropoff_census_tract              0
pickup_community_area          7538
dropoff_community_area        80242
fare                              0
tips                              0
tolls                             0
extras                            0
trip_total                        0
payment_type                      0
company                           0
pickup_centroid_latitude          0
pickup_centroid_longitude         0
dropoff_centroid_latitude         0
dropoff_centroid_longitude        0
hour                            104
dayofweek                       104
month                           104
trip_minutes                      0
tip_pct                       16681
year                            104
day                         

Most of the missing values are the community area. As the census tract corresponds with the community we can use the mapping from census tract and community area, which is available online to fill in the missing values. Also some timestamps are missing.

### Missing Timestamps
First we try to identify if the missing timestamps came from parsing them to the local time in Chicago.

In [9]:
RAW_DATA_DIR = ROOT_DIR / "data" / "raw"
raw_ts = pd.read_csv(RAW_DATA_DIR / "chicago_taxi_trips_2024.csv", 
                      usecols=["trip_start_timestamp"])
nat_rows = df[df["trip_start_timestamp"].isna()].index
print(raw_ts.loc[nat_rows, "trip_start_timestamp"].unique())

<StringArray>
['2024-11-03T01:00:00.000', '2024-11-03T01:15:00.000',
 '2024-11-03T01:30:00.000', '2024-11-03T01:45:00.000',
 '2025-11-02T01:00:00.000', '2025-11-02T01:15:00.000',
 '2025-11-02T01:30:00.000', '2025-11-02T01:45:00.000']
Length: 8, dtype: str


The missing timestamps coincide with the daylight saving time transition from summer to winter time. Rather than being truly absent, these entries were simply dropped during parsing because they were ambiguous or invalid in that context. AS this is the case for a very small portion of trips, we can just drop these observations.

### Missing Community Areas
Chicago census tracts are fully nested within community areas, so every trip coordinate maps to exactly one community area. We use a spatial join against the official Chicago community area boundaries (downloaded via `scripts/download_community_areas.py`) to fill the missing values. Trips whose coordinates fall outside Chicago's 77 community areas (e.g. suburban drop-offs) will remain NaN and are kept as-is.

### Missing taxiID

In [13]:
print(df.isnull().sum())

trip_id                           0
taxi_id                           0
trip_start_timestamp              0
trip_end_timestamp                0
trip_seconds                      0
trip_miles                        0
pickup_census_tract               0
dropoff_census_tract              0
pickup_community_area             0
dropoff_community_area            0
fare                              0
tips                              0
tolls                             0
extras                            0
trip_total                        0
payment_type                      0
company                           0
pickup_centroid_latitude          0
pickup_centroid_longitude         0
dropoff_centroid_latitude         0
dropoff_centroid_longitude        0
hour                              0
dayofweek                         0
month                             0
trip_minutes                      0
tip_pct                       16680
year                              0
day                         

## Consistency Checks

### Trips that didn't happen
Are the any trips where the start and end is at the same time and the pickup and dropoff is at the same place? These trips can be savely deleted as they don't produce any value and don't count as a real trip for our demand.

In [14]:
mask = (
    (df['trip_seconds'] == 0) &
    (df['trip_end_timestamp'] == df['trip_start_timestamp']) &
    (df['pickup_centroid_latitude'] == df['dropoff_centroid_latitude']) &
    (df['pickup_centroid_longitude'] == df['dropoff_centroid_longitude'])
)
mask.sum()





np.int64(83220)

### Trips with 0 miles

We will check for trips with 0 miles but where the duration of the trip is longer than 0 seconds

In [16]:
df_zero_miles = df[(df['trip_miles'] == 0) & (df['trip_seconds'] > 0)]
print(df_zero_miles['trip_seconds'].describe())

count    496056.000000
mean        478.144419
std        1399.267550
min           1.000000
25%           6.000000
50%          46.000000
75%         511.000000
max       86396.000000
Name: trip_seconds, dtype: float64


In [17]:
mask_2 = (
    df['trip_miles'] == 0
)
mask_2.sum()

np.int64(497925)

In [18]:
mask_2 = (
    (df['trip_miles'] <= 0) &
    (df['trip_seconds'] > 0)
)
mask_2.sum()


np.int64(496056)

In [19]:
mask_3 = (
    df['trip_end_timestamp'] < df['trip_start_timestamp']
)
mask_3.sum()

np.int64(0)

## Handle Duplicates

### Check for duplicated trips on tridId

In [20]:
dup_df = df[df['trip_id'].duplicated(keep=False)]
print(f"Number of duplicate trip_id entries:", dup_df.shape[0])

Number of duplicate trip_id entries: 0


## Loading & Preprocessing in Subsequent Notebooks

All steps documented above are encapsulated in two helper scripts so that later notebooks can load a clean dataset in a single line:

- **`scripts/datasets.py`** — entry point for loading any dataset. `load_taxi_data(preprocessed=True)` reads the raw CSV and optionally applies the full preprocessing pipeline.
- **`scripts/preprocessing.py`** — contains `preprocess_taxi_data()`, which adds temporal features, imputes missing community areas, drops invalid rows, and returns a clean `DataFrame`.

Usage in any subsequent notebook:

```python
from scripts.datasets import load_taxi_data

df = load_taxi_data()                    # cleaned dataset
df_raw = load_taxi_data(preprocessed=False)  # raw data only
```